# Analysis of experimental results

In this notebook we analyse the loss landscape captured during the experiments and verify that the reparam checkpoints are indeed flatter than the baseline checkpoints.

## Method of experimentation

Dataset: CIFAR-10
Models: ResNet-18 and ViT-B/32
Optimizers: SGD, SAM, ASAM, M-SAM

We train ResNet-18 model on CIFAR-10 for 200 epochs with SGD, SAM, ASAM, and M-SAM optimizers and fine-tune ViT-B/32 on CIFAR-10 for 50 epochs with the same optimizers. During the training, we capture following metrics:

- Model checkpoint
- training loss and accuracy
- test loss and accuracy
- seed
- optimizer type and hyperparameters (e.g. rho for SAM)

### Statistical analysis of metrics

We track divergence rate $\delta = \mathcal{L}_{test} - \mathcal{L}_{train}$ and Test accuracy against the hyperparameter $\rho$ for SAM-based optimizers. 
$\rho$ controls the size of the neighborhood considered for sharpness-aware optimization, and we expect that higher $\rho$ values will lead to flatter minima and thus lower divergence rates. We will analyze the relationship between $\delta$ and $\rho$ to verify this hypothesis.
However, we also note that the relationship between $\delta$ and $\rho$ may not be strictly monotonic, as excessively high $\rho$ values can lead to optimization instability and worse generalization.

In this experiment we sweep $\rho$ for SAM-based optimizers in the range $\{0.05, 0.1, 0.5\}$ and track the corresponding $\delta$ values to analyze this relationship.

## Flatness analysis of checkpoints

Plot flatness metrics for the checkpoints and compare between baseline and reparam models to verify that the reparametrized models are indeed flatter than the baseline models. Follow the same implmentation of plotting loss landscape as in the `run_flatness.py` script.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torchvision.models as tvm
from torchvision.models import ViT_B_32_Weights
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tqdm

import copy

seed = 42
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(seed)

# ── repo root on sys.path ────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path("/Users/manavdahra/workspace/sam-opt-research")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# data dir for cifar10
DATA_DIR = os.path.join(ROOT, "data/cifar10")

MODEL_TYPE = "vit_b_32"
OPTIMIZER = "asam"
RHO = 0.5
ALPHA = 5.0

# The checkpoints are saved under the directory `results_per_instance` for now

MODEL_DATA_DIRS = {
    "resnet18": {
        "baseline": "resnet18/experiments/baseline/resnet18",
        "reparam": "resnet18/experiments/reparam/resnet18"
    },
    "vit_b_32": {
        "baseline": "vit_b_32/experiments/baseline/vit_b_32",
        "reparam": "vit_b_32/experiments/reparam/vit_b_32"
    }
}

BASELINE_CHKPT_DIR = os.path.join(ROOT, f"results/{MODEL_DATA_DIRS[MODEL_TYPE]['baseline']}/checkpoints")
REPARAM_CHKPT_DIR = os.path.join(ROOT, f"results/{MODEL_DATA_DIRS[MODEL_TYPE]['reparam']}/checkpoints")

BASELINE_CHKPT_NAME = f"{OPTIMIZER}_rho{RHO}_seed{seed}.pt"
REPARAM_CHKPT_NAME = f"{OPTIMIZER}_rho{RHO}_alpha{ALPHA}_seed{seed}.pt"

BASELINE_CHKPT = os.path.join(BASELINE_CHKPT_DIR, BASELINE_CHKPT_NAME)
REPARAM_CHKPT = os.path.join(REPARAM_CHKPT_DIR, REPARAM_CHKPT_NAME)

/Users/manavdahra/workspace/sam-opt-research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Models

We use ResNet-18 and ViT-B/32 models for our experiments. ResNet-18 is a widely used convolutional neural network architecture that has been shown to perform well on image classification tasks, while ViT-B/32 is a vision transformer model that has recently achieved state-of-the-art results on various vision benchmarks. By using these two different architectures, we can analyze the effect of reparametrization on both convolutional and transformer-based models, providing a more comprehensive understanding of the impact of reparametrization on model flatness and generalization.

For ViT-B/32, we use the pretrained weights from the `timm` library and fine-tune on CIFAR-10, while for ResNet-18 we train from scratch.

In [2]:
from src.models.resnet18 import get_resnet18
from src.models.vit import get_vit_b_32

def build_model(model_name, device: torch.device) -> nn.Module:
    if model_name == "resnet18":
        model = get_resnet18(num_classes=10)
    elif model_name == "vit_b_32":
        model = get_vit_b_32(num_classes=10, pretrained=True)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model.to(device)

## Load model from the checkpoints


In [3]:
def load_model_from_checkpoint(checkpoint_path):
    m = build_model(MODEL_TYPE.lower(), device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    print(f"Loaded checkpoint from {checkpoint_path}, keys: {list(state_dict.keys())[:5]} ...")
    m.load_state_dict(state_dict)
    return m

baseline_model = load_model_from_checkpoint(BASELINE_CHKPT)
reparam_model = load_model_from_checkpoint(REPARAM_CHKPT)


Loaded checkpoint from /Users/manavdahra/workspace/sam-opt-research/results/vit_b_32/experiments/baseline/vit_b_32/checkpoints/asam_rho0.5_seed42.pt, keys: ['class_token', 'conv_proj.weight', 'conv_proj.bias', 'encoder.pos_embedding', 'encoder.layers.encoder_layer_0.ln_1.weight'] ...
Loaded checkpoint from /Users/manavdahra/workspace/sam-opt-research/results/vit_b_32/experiments/reparam/vit_b_32/checkpoints/asam_rho0.5_alpha5.0_seed42.pt, keys: ['class_token', 'conv_proj.weight', 'conv_proj.bias', 'encoder.pos_embedding', 'encoder.layers.encoder_layer_0.ln_1.weight'] ...


Lets spot check if the model parameters are indeed different because of reparametrization. We will compare the weights of parameters across each layer and print name of the layers that differ.

## Sharpness analysis of checkpoints

### Technique: Hutchinson trace estimate

$$\text{tr}(H) \approx \frac{1}{n} \sum_{i=1}^n z_i^T H z_i$$

Where $H$ is the Hessian of the loss function with respect to model parameters, and $z_i$ are random vectors sampled from a Rademacher distribution (i.e., each element is independently set to +1 or -1 with equal probability). The trace of the Hessian, $\text{tr}(H)$, serves as a proxy for sharpness, with higher values indicating sharper minima. By comparing the Hutchinson trace estimates for baseline and reparametrized models, we can quantitatively assess the difference in flatness between them.

In [4]:
from src.analysis.hutchinson import hutchinson_trace

## Plotting 2D-Loss landscape

We plot the 2D loss landscape along two random filter-normalized directions for both baseline and reparametrized models. The loss landscape will be visualized using contour plots, where the x and y axes represent the perturbation magnitudes along the two directions, and the color intensity represents the loss value. By comparing the loss landscapes of baseline and reparametrized models, we can visually assess the difference in flatness between them, with flatter landscapes indicating better generalization potential.

In [5]:
from src.analysis.landscape import loss_landscape_2d, plot_loss_landscape_2d


In [11]:
from src.data.cifar10 import get_cifar10_loaders

def analyse(
        name: str,
        model: nn.Module,
        batch_size: int = 256,
        num_workers: int = 4,
        max_samples: int = 5000,
        steps: int = 31,
        range_: float = 1.0,
        resize: int | None = None,
) -> None:
    _, test_loader = get_cifar10_loaders(
        data_dir=DATA_DIR,
        batch_size=batch_size,
        num_workers=num_workers,
        resize=resize,
        max_samples=max_samples,
        shuffle=False,
    )
    loss_fn = nn.CrossEntropyLoss()

    print(f"\nAnalysing: {name}")
    trace, alphas, betas, losses = loss_landscape_2d(
        model, loss_fn, test_loader, device,
        steps=steps, range_=range_,
    )

    fig = plot_loss_landscape_2d(
        np.array(alphas),
        np.array(betas),
        np.array(losses),
        title=f"2D Loss Landscape for {name}, Tr(H)/d = {trace:.6f}",
    )
    fig.show()


In [13]:
analyse(
    name=BASELINE_CHKPT_NAME,
    model=baseline_model,
    batch_size=128,
    num_workers=4,
    max_samples=1280,    # match max_batch × batch_size so no unused data is loaded
    steps=31,           # 21×21=441 vs 31×31=961 grid points: ~2× faster, still smooth surface
    resize=224 if MODEL_TYPE.lower() == "vit_b_32" else None,
    range_=30.0,
)



Analysing: asam_rho0.5_seed42.pt


ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
analyse(
    name=REPARAM_CHKPT_NAME,
    model=reparam_model,
    batch_size=128,
    num_workers=4,
    n_samples=20,       # 10 vs 20: Hutchinson is already stochastic; 10 is sufficient for a relative comparison
    max_batch=1,        # 10 batches × 128 = 1280 samples: stable loss estimate, half the landscape cost
    max_samples=1280,    # match max_batch × batch_size so no unused data is loaded
    steps=31,           # 21×21=441 vs 31×31=961 grid points: ~2× faster, still smooth surface
    resize=224 if MODEL_TYPE.lower() == "vit_b_32" else None,
    range_=30.0,
)


## Hessian eigenspectrum

We approximate the top-$k$ eigenvalues of the Hessian $H$ using the **Lanczos algorithm**, which builds a low-dimensional Krylov subspace via repeated Hessian-vector products (each computed in $\mathcal{O}(d)$ via two backward passes — the Pearlmutter trick). This avoids materializing the full $d \times d$ Hessian ($d \approx 11M$ for ResNet-18).

After $k$ Lanczos steps the Hessian is projected onto a tridiagonal matrix $T \in \mathbb{R}^{k \times k}$, whose eigenvalues approximate the extreme eigenvalues of $H$. We compare the spectra of the baseline and reparametrized checkpoints: **sharper minima → larger top eigenvalues**.


In [ ]:
from scipy.sparse.linalg import eigsh, LinearOperator


def _hvp(model, loss_fn, inputs, targets, v_tensors):
    """Hessian-vector product H @ v via two backward passes (Pearlmutter trick).

    Args:
        v_tensors: list of tensors matching model.parameters() shapes.

    Returns:
        Tuple of tensors (one per parameter) representing H @ v.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    outputs = model(inputs)
    loss = loss_fn(outputs, targets)
    grads = torch.autograd.grad(loss, params, create_graph=True)
    grad_v = sum((g * vi).sum() for g, vi in zip(grads, v_tensors))
    hv = torch.autograd.grad(grad_v, params, retain_graph=False)
    return hv


def hessian_eigenvalues(
    model: nn.Module,
    loss_fn: nn.Module,
    loader: DataLoader,
    device: torch.device,
    k: int = 50,
    max_batch: int = 64,
    which: str = "LM",
) -> np.ndarray:
    """Estimate the top-k Hessian eigenvalues via ARPACK (scipy eigsh).

    Wraps the Hessian-vector product as a scipy LinearOperator so ARPACK
    handles the Krylov subspace construction, re-orthogonalization, and
    convergence — much more numerically robust than a hand-rolled Lanczos.

    Args:
        k: Number of eigenvalues to compute.
        max_batch: Number of images used for each Hessian-vector product.
        which: Which eigenvalues to target — 'LM' (largest magnitude),
               'SM' (smallest magnitude), 'LA' (largest algebraic), etc.

    Returns:
        eigenvalues: 1D numpy array of shape (k,), sorted ascending.
    """
    model.eval()
    inputs, targets = next(iter(loader))
    inputs = inputs[:max_batch].to(device)
    targets = targets[:max_batch].to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    shapes = [p.shape for p in params]
    sizes  = [p.numel() for p in params]
    d = sum(sizes)

    def _matvec(v_np: np.ndarray) -> np.ndarray:
        """scipy callback: flat float32 numpy → flat float32 numpy."""
        # Reconstruct list of tensors matching each parameter
        v_tensors = []
        idx = 0
        for p, s in zip(params, sizes):
            chunk = torch.tensor(v_np[idx: idx + s], dtype=torch.float32, device=device).reshape(p.shape)
            v_tensors.append(chunk)
            idx += s

        hv = _hvp(model, loss_fn, inputs, targets, v_tensors)

        # Flatten and move to CPU numpy
        return np.concatenate([h.detach().cpu().to(torch.float32).numpy().ravel() for h in hv])

    A = LinearOperator((d, d), matvec=_matvec, dtype=np.float32)

    # k must be < d; for large models this is never an issue
    actual_k = min(k, d - 1)
    eigenvalues, _ = eigsh(A, k=actual_k, which=which, tol=1e-3, maxiter=300)
    return np.sort(eigenvalues)


def plot_eigenspectrum(
    eigenvalues_dict: dict[str, np.ndarray],
    title: str = "Hessian Eigenspectrum",
    log_scale: bool = False,
) -> go.Figure:
    """Overlay eigenvalue spectra for multiple models.

    Args:
        eigenvalues_dict: Mapping from label → 1D array of eigenvalues.
        log_scale: If True, use log scale on the y-axis.
    """
    fig = go.Figure()
    colors = ["steelblue", "tomato", "mediumseagreen", "darkorange"]
    for (label, eigs), color in zip(eigenvalues_dict.items(), colors):
        fig.add_trace(go.Bar(
            x=list(range(len(eigs))),
            y=eigs.tolist(),
            name=label,
            marker_color=color,
            opacity=0.75,
        ))
    fig.update_layout(
        title=title,
        xaxis_title="Eigenvalue index (sorted ascending)",
        yaxis_title="Eigenvalue",
        yaxis_type="log" if log_scale else "linear",
        barmode="overlay",
        template="plotly_white",
        legend=dict(orientation="h", y=-0.2),
    )
    return fig


In [ ]:
_, test_loader = get_cifar10_loaders(
    data_dir=DATA_DIR,
    batch_size=256,
    max_samples=2560,
    shuffle=False,
    resize=224 if MODEL_TYPE.lower() == "vit_b_32" else None,
)
loss_fn = nn.CrossEntropyLoss()

print("Computing Hessian eigenspectrum for baseline model (ARPACK, k=50)...")
baseline_eigs = hessian_eigenvalues(baseline_model, loss_fn, test_loader, device, k=50, max_batch=64)
print(f"  Top-5 eigenvalues: {baseline_eigs[-5:]}")

print("Computing Hessian eigenspectrum for reparam model (ARPACK, k=50)...")
reparam_eigs = hessian_eigenvalues(reparam_model, loss_fn, test_loader, device, k=50, max_batch=64)
print(f"  Top-5 eigenvalues: {reparam_eigs[-5:]}")

fig = plot_eigenspectrum(
    {
        f"Baseline ({BASELINE_CHKPT_NAME})": baseline_eigs,
        f"Reparam  ({REPARAM_CHKPT_NAME})":  reparam_eigs,
    },
    title="Hessian Eigenspectrum — Baseline vs Reparam (ARPACK eigsh, k=50)",
    log_scale=False,
)
fig.show()
